Marko Saar (Product Manager, UrbanStyle.ltd) 
"We have a new customer dataset. I want to know: who are our VIP customers? How much do they spend? And why do some stop buying?
The dashboard shows aggregated data — but I need individual, customer-level analysis. I want to send different emails to VIP customers and at-risk customers.

In [ ]:
from supabase import create_client, Client

In [ ]:
import pandas as pd
from supabase import create_client, Client

SUPABASE_URL = "supabase url"
SUPABASE_KEY = "public anon"

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

print("Supabase connected!")

In [ ]:
# Load the raw sales transactions dataset
df_raw = pd.read_csv("urbanstyle_sales.csv")

# Inspect dataset shape and column structure
print("Dataset Shape:", df_raw.shape)
df_raw.info()
df_raw.head()

In [ ]:
# Create a working copy
df = df_raw.copy()

# Remove duplicate transaction records if present
df = df.drop_duplicates()

# Handle missing customer identifiers or null fields
df = df.dropna(subset=['customer_id'])

# Convert sale date to datetime object
df['sale_date'] = pd.to_datetime(df['sale_date'])

# Ensure positive non-zero quantities and unit prices
df = df[(df['quantity'] > 0) & (df['unit_price'] > 0)]

# Calculate line-item total spend in EUR
df['total_price'] = df['quantity'] * df['unit_price']

print("Cleaned Dataset Shape:", df.shape)
df.head()

In [ ]:
# Set snapshot reference date for recency calculation
today = pd.to_datetime('2025-02-28')

# Aggregate metrics per unique customer
rfm = df.groupby('customer_id').agg(
    last_purchase=('sale_date', 'max'),
    frequency=('sale_id', 'count'),
    monetary=('total_price', 'sum')
).reset_index()

# Calculate Recency in days
rfm['recency_days'] = (today - rfm['last_purchase']).dt.days

# Generate standard quartile scores (1-4 scale)
rfm['R_quartile'] = pd.qcut(rfm['recency_days'], q=4, labels=[4, 3, 2, 1]).astype(int)
rfm['F_quartile'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4]).astype(int)
rfm['M_quartile'] = pd.qcut(rfm['monetary'], q=4, labels=[1, 2, 3, 4]).astype(int)

# Simple average RFM Score for standard tiering
rfm['RFM_Score'] = rfm[['R_quartile', 'F_quartile', 'M_quartile']].mean(axis=1)

# Map standard 5 segments
def map_standard_segment(score):
    if score >= 3.5:
        return 'VIP Champions'
    elif score >= 2.8:
        return 'Loyal'
    elif score >= 2.2:
        return 'Potential'
    elif score >= 1.5:
        return 'At Risk'
    else:
        return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(map_standard_segment)
rfm.head()

In [ ]:
# Define consistent color palette
color_map = {
    'VIP Champions': '#2a9d8f',
    'Loyal': '#e9c46a',
    'Potential': '#f4a261',
    'At Risk': '#e76f51',
    'Lost': '#d62828'
}

# -------------------------------------------------------------------------
# CHART 1: Customer Count by Segment
# -------------------------------------------------------------------------
segment_counts = rfm['Segment'].value_counts().reindex(
    ['VIP Champions', 'Loyal', 'Potential', 'At Risk', 'Lost']
).reset_index()
segment_counts.columns = ['Segment', 'Customer Count']

fig1 = px.bar(
    segment_counts,
    x='Segment',
    y='Customer Count',
    color='Segment',
    color_discrete_map=color_map,
    title='UrbanStyle Customer Count by Segment',
    labels={'Customer Count': 'Number of Customers', 'Segment': 'Customer Segment'},
    text_auto=True
)
fig1.update_layout(showlegend=False, template='plotly_white')
fig1.show()

# -------------------------------------------------------------------------
# CHART 2: Recency vs Monetary Scatter
# -------------------------------------------------------------------------
fig2 = px.scatter(
    rfm,
    x='recency_days',
    y='monetary',
    color='Segment',
    size='frequency',
    color_discrete_map=color_map,
    title='UrbanStyle Customer Segments: Recency vs Spend',
    labels={
        'recency_days': 'Days Since Last Purchase (Recency)',
        'monetary': 'Total Spend in EUR (Monetary)',
        'frequency': 'Order Count',
        'Segment': 'Segment'
    },
    hover_data=['customer_id']
)
fig2.update_layout(template='plotly_white')
fig2.show()

# -------------------------------------------------------------------------
# CHART 3: Top 10 VIP Champions
# -------------------------------------------------------------------------
top_10_vips = rfm[rfm['Segment'] == 'VIP Champions'].nlargest(10, 'monetary').sort_values('monetary', ascending=True)

fig3 = px.bar(
    top_10_vips,
    x='monetary',
    y='customer_id',
    orientation='h',
    title='Top 10 VIP Champions by Total Spend',
    labels={'monetary': 'Total Spend (EUR)', 'customer_id': 'Customer ID'},
    text_auto='.2f'
)
fig3.update_traces(marker_color='#2a9d8f')
fig3.update_layout(template='plotly_white', yaxis=dict(type='category'))
fig3.show()

# -------------------------------------------------------------------------
# PRINT EXECUTIVE SUMMARY METRICS
# -------------------------------------------------------------------------
total_rev = rfm['monetary'].sum()
vip_rev = rfm[rfm['Segment'] == 'VIP Champions']['monetary'].sum()
vip_share = (vip_rev / total_rev) * 100
vip_count = len(rfm[rfm['Segment'] == 'VIP Champions'])
at_risk_count = len(rfm[rfm['Segment'] == 'At Risk'])

print("=== DATA INSIGHTS FOR MARKO ===")
print(f"Total Revenue: {total_rev:,.2f} EUR")
print(f"VIP Champions Count: {vip_count} customers")
print(f"VIP Champions Revenue Share: {vip_share:.2f}%")
print(f"At-Risk Customers: {at_risk_count} customers")